In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# importing the libraries
import pandas as pd
import numpy as np
from pathlib import Path
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

In [ ]:
# setting up the paths and importing the dataset
DATA_DIR = Path("Data download/data")

INPUT_PATH = DATA_DIR / "filings_clean.csv"
OUTPUT_PATH = DATA_DIR / "filings_finbert.csv"
CHECKPOINT_PATH = DATA_DIR / "finbert_checkpoint.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} filings")
print(f"Downside rate: {df['downside'].mean():.1%}")

In [ ]:
print(f"Loaded {len(df)} filings")
print(f"Downside rate: {df['downside'].mean():.1%}")
print(f"Median text len: {df['cleanTextLen'].median():.0f}")
# Must print: 5280 | 25.0% | ~14908

In [ ]:
MODEL_NAME = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model = model.to(device)

# yiyanghkust/finbert-tone uses 'Negative' with capital N
label_values = list(model.config.id2label.values())
NEGATIVE_IDX = next(i for i, v in enumerate(label_values) if v.lower() == 'negative')

print(f"Device: {device}")
print(f"Label mapping: {model.config.id2label}")
print(f"Negative index: {NEGATIVE_IDX}")

In [ ]:
# Forward-looking keywords — sentences containing these get 3x weight
# because they carry the most market-relevant signal in earnings releases
_FLS_KEYWORDS = re.compile(
    r'\b(expect|guid(e|ance)|outlook|forecast|anticipat|project(ion)?|'
    r'target|full[- ]year|next quarter|fiscal 20\d\d|going forward|'
    r'rais(e|ing)|lower(ing)?|reaffirm|withdraw|suspend)\b',
    re.IGNORECASE
)

def score_finbert(text: str, batch_size: int = 32) -> dict:
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {'finbert_neg_mean': np.nan,
                'finbert_neg_weighted': np.nan,
                'finbert_n_sentences': 0}

    # Better sentence splitting:
    # 1. Split on .!? followed by whitespace + capital letter
    #    (avoids splitting on decimals like $2.50 or abbreviations)
    # 2. Also split on newlines which often separate press release sections
    raw = re.split(r'(?<=[.!?])\s+(?=[A-Z])|(?<=\n)', text)
    sentences = [s.strip() for s in raw if len(s.strip()) > 30]  # raised from 10 to 30

    if not sentences:
        return {'finbert_neg_mean': np.nan,
                'finbert_neg_weighted': np.nan,
                'finbert_n_sentences': 0}

    neg_probs = []
    weights   = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs  = F.softmax(logits, dim=-1)

        neg_probs.extend(probs[:, NEGATIVE_IDX].cpu().tolist())

        # Weight forward-looking sentences 3x, all others 1x
        for s in batch:
            weights.append(3.0 if _FLS_KEYWORDS.search(s) else 1.0)

    neg_probs = np.array(neg_probs)
    weights   = np.array(weights)

    return {
        'finbert_neg_mean'    : float(np.mean(neg_probs)),           # original metric — keep for comparison
        'finbert_neg_weighted': float(np.average(neg_probs, weights=weights)),  # new weighted metric
        'finbert_n_sentences' : len(sentences)
    }

In [ ]:
# Quick test before full run
sample = df['cleanText'].iloc[0]
print(score_finbert(sample))

In [ ]:
CHECKPOINT_PATH = DATA_DIR / "finbert_checkpoint.csv"
CHECKPOINT_EVERY = 100

# Resume from checkpoint if it exists
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    done = set(checkpoint_df['accessionNumber'].tolist())
    results = checkpoint_df.to_dict(orient='records')
    print(f"Resuming from checkpoint: {len(done)} filings already scored")
else:
    done = set()
    results = []
    print("No checkpoint found, starting from scratch")

# Score remaining filings
remaining = [(idx, row) for idx, row in df.iterrows()
             if row['accessionNumber'] not in done]
print(f"Filings remaining: {len(remaining)}")

for i, (idx, row) in enumerate(tqdm(remaining, desc="FinBERT scoring")):
    try:
        result = score_finbert(row['cleanText'])
        result['accessionNumber'] = row['accessionNumber']
        result['status'] = 'ok'
    except Exception as e:
        result = {
            'accessionNumber'     : row['accessionNumber'],
            'finbert_neg_mean'    : np.nan,
            'finbert_neg_weighted': np.nan,
            'finbert_n_sentences' : 0,
            'status'              : str(e)
        }
    results.append(result)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
        tqdm.write(f"Checkpoint saved at {len(results)} filings")

# Final checkpoint save
scores_df = pd.DataFrame(results)
scores_df.to_csv(CHECKPOINT_PATH, index=False)

# Merge back into df on accessionNumber
df = df.merge(
    scores_df[['accessionNumber', 'finbert_neg_mean',
               'finbert_neg_weighted', 'finbert_n_sentences']],
    on='accessionNumber', how='left'
)

# Diagnostics
print(f"\nScored ok : {(scores_df['status']=='ok').sum()}")
print(f"Failed    : {(scores_df['status']!='ok').sum()}")
print(f"\nSentence count distribution:")
print(df['finbert_n_sentences'].describe().round(0))
print(f"Filings with >500 sentences: {(df['finbert_n_sentences'] > 500).sum()}")
print(f"\nNull finbert_neg_mean: {df['finbert_neg_mean'].isna().sum()}")
print(f"\nMean finbert_neg_mean downside=1 : {df[df['downside']==1]['finbert_neg_mean'].mean():.4f}")
print(f"Mean finbert_neg_mean downside=0 : {df[df['downside']==0]['finbert_neg_mean'].mean():.4f}")

In [ ]:
# Save FinBERT scores with accessionNumber as merge key
FINBERT_CHECKPOINT_PATH = DATA_DIR / "finbert_checkpoint.csv"

df[['accessionNumber', 'ticker', 'filingDate', 'car_0_1', 'downside',
    'finbert_neg_mean', 'finbert_neg_weighted', 'finbert_n_sentences']].to_csv(
    FINBERT_CHECKPOINT_PATH, index=False
)

print(f"Saved to {FINBERT_CHECKPOINT_PATH}")
print(f"Rows: {len(df)}")
print(f"Sample accessionNumbers: {df['accessionNumber'].head(3).tolist()}")

In [ ]:
from sklearn.metrics import roc_auc_score

df_eval = df.dropna(subset=['finbert_neg_mean'])
print(f"Filings evaluated: {len(df_eval)}")

auc_mean     = roc_auc_score(df_eval['downside'], df_eval['finbert_neg_mean'])
auc_weighted = roc_auc_score(df_eval['downside'], df_eval['finbert_neg_weighted'])
print(f"FinBERT AUC (simple mean)    : {auc_mean:.4f}")
print(f"FinBERT AUC (weighted mean)  : {auc_weighted:.4f}")

for col in ['finbert_neg_mean', 'finbert_neg_weighted']:
    d1 = df_eval[df_eval['downside']==1][col].mean()
    d0 = df_eval[df_eval['downside']==0][col].mean()
    print(f"\n{col}:")
    print(f"  downside=1 : {d1:.6f}")
    print(f"  downside=0 : {d0:.6f}")
    print(f"  difference : {d1-d0:.6f}")

decile_threshold = df_eval['finbert_neg_weighted'].quantile(0.9)
bottom_decile    = df_eval[df_eval['finbert_neg_weighted'] >= decile_threshold]
print(f"\nBottom decile avg CAR (weighted): {bottom_decile['car_0_1'].mean():.4f}")

In [ ]:
print(f"Mean finbert_neg_mean  downside=1 : {df[df['downside']==1]['finbert_neg_mean'].mean():.4f}")
print(f"Mean finbert_neg_mean  downside=0 : {df[df['downside']==0]['finbert_neg_mean'].mean():.4f}")
print(f"\nScore distribution:")
print(df['finbert_neg_mean'].describe().round(4))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np

# Load FinBERT scores and GPT accession numbers
finbert_df = pd.read_csv(DATA_DIR / "finbert_checkpoint.csv")
gpt_df     = pd.read_csv(DATA_DIR / "gpt_checkpoint.csv")

# Filter to same 2000 filings as GPT
finbert_sub = finbert_df[finbert_df['accessionNumber'].isin(gpt_df['accessionNumber'])].copy()
print(f"FinBERT subsample : {len(finbert_sub)} filings")
print(f"Downside          : {finbert_sub['downside'].sum()}")
print(f"Non-downside      : {(finbert_sub['downside']==0).sum()}")

# Same 80/20 split, same random seed as GPT
val_df, test_df = train_test_split(
    finbert_sub,
    test_size=0.80,
    stratify=finbert_sub['downside'],
    random_state=42
)

print(f"\nValidation set    : {len(val_df)} filings")
print(f"Test set          : {len(test_df)} filings")

# Full sample AUC
auc_full_mean     = roc_auc_score(finbert_df['downside'], finbert_df['finbert_neg_mean'])
auc_full_weighted = roc_auc_score(finbert_df['downside'], finbert_df['finbert_neg_weighted'])
print(f"\nFinBERT AUC simple mean   (full, n=5280)     : {auc_full_mean:.4f}")
print(f"FinBERT AUC weighted mean (full, n=5280)     : {auc_full_weighted:.4f}")

# Subsample AUC
auc_sub_mean     = roc_auc_score(finbert_sub['downside'], finbert_sub['finbert_neg_mean'])
auc_sub_weighted = roc_auc_score(finbert_sub['downside'], finbert_sub['finbert_neg_weighted'])
print(f"\nFinBERT AUC simple mean   (subsample, n=2000): {auc_sub_mean:.4f}")
print(f"FinBERT AUC weighted mean (subsample, n=2000): {auc_sub_weighted:.4f}")

# Test set AUC
auc_test_mean     = roc_auc_score(test_df['downside'], test_df['finbert_neg_mean'])
auc_test_weighted = roc_auc_score(test_df['downside'], test_df['finbert_neg_weighted'])
print(f"\nFinBERT AUC simple mean   (test set, n=1600) : {auc_test_mean:.4f}")
print(f"FinBERT AUC weighted mean (test set, n=1600) : {auc_test_weighted:.4f}")

# Bootstrap CI on test set (weighted mean as primary)
np.random.seed(42)
boot_aucs = []
for _ in range(1000):
    sample = test_df.sample(len(test_df), replace=True)
    try:
        boot_aucs.append(roc_auc_score(sample['downside'], sample['finbert_neg_weighted']))
    except:
        pass

ci_low  = np.percentile(boot_aucs, 2.5)
ci_high = np.percentile(boot_aucs, 97.5)
print(f"\n95% bootstrap CI (weighted, test set): [{ci_low:.4f}, {ci_high:.4f}]")